# 🎯 大語言模型激活值校準系統 (Google Colab 雙帳號加速版)

本筆記本提供在 **Google Colab (T4 / L4 / A100 GPU)** 上一鍵運行大語言模型（LLM）的**解耦式激活值校準（Decoupled Activation Calibration）**。

### 💡 核心優勢與工作流程
1. **解耦架構 (Decoupled 2-Stage)**：校準只需執行一次，統計量（`Scale`, `Zero Point`, `99.99% Quantile Clip`, `Channel Max`）會打包儲存為 `.pt` 檔案至 Google Drive。後續測試 10 種量化組合（W8A16, W4A16, W4A4, SpinQuant 等）可**秒級載入，無需重新跑前向傳播**！
2. **雙帳號平行加速 (~35 分鐘完成全部 5 個模型)**：
   * **Account 1**：Llama 3.2 1B (`~6m`) ➔ Gemma 2 2B (`~8m`) ➔ Llama 3.2 3B (`~17m`) ➔ **累計 ~31 分鐘**
   * **Account 2**：Qwen 3.5 2B (`~11m`) ➔ Qwen 3.5 4B (`~23m`) ➔ **累計 ~34 分鐘**
3. **學術標準規格**：嚴格遵循 GPTQ / AWQ / QuaRot 標準，採集 `wikitext-2-raw-v1` `train` 分割集之 **128 條 × 2048 tokens**（共 262,144 tokens，`seed=42`）。

---

## 1. 檢查 GPU 資源 (NVIDIA-SMI)

In [ ]:
!nvidia-smi

## 2. 掛載 Google Drive (自動持久化保存校準 .pt 檔案)
將校準產出的權重檔直接寫入 Google Drive，避免 Colab 閒置斷線遺失數據。

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 建立校準輸出目錄
CALIB_OUTPUT_DIR = "/content/drive/MyDrive/model_quantization/calibration_results"
os.makedirs(CALIB_OUTPUT_DIR, exist_ok=True)
print(f"✅ 校準結果儲存目錄就緒: {CALIB_OUTPUT_DIR}")

## 3. Clone 專案倉庫與安裝必要依賴

In [ ]:
import os

# 請填寫您的 GitHub 倉庫 URL (例如 https://github.com/your-username/model_quantization.git)
REPO_URL = "https://github.com/your-username/model_quantization.git"

if not os.path.exists("/content/model_quantization"):
    !git clone $REPO_URL /content/model_quantization
    %cd /content/model_quantization
else:
    %cd /content/model_quantization
    !git pull

# 安裝核心依賴
!pip install -q "transformers>=4.40.0" "accelerate>=0.28.0" "datasets>=2.18.0" sentencepiece protobuf scipy tqdm

## 4. Hugging Face 權限驗證 (存取 Llama 3.2 與 Gemma 2 必備)
若是執行 Qwen 系列模型可跳過此步驟；若是 Llama 或 Gemma，推薦透過 Colab 左側鑰匙圖示 (Secrets) 安全讀取，杜絕任何代碼暴露。

In [ ]:
from huggingface_hub import login

# 推薦安全方式：在 Colab 左側 Secrets 新增名為 'HF_TOKEN' 的變數
try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
    print("✅ 成功透過 Colab Secrets 安全認證 Hugging Face！")
except Exception:
    # 若未在 Secrets 設定，可取消註解透過互動式彈窗輸入：
    # login()
    pass

---
## 🚀 任務排程方案 A：帳號 1 執行 (Account 1)
**目標模型**：`Llama-3.2-1B` ➔ `gemma-2-2b-it` ➔ `Llama-3.2-3B`
**總耗時預估**：約 **31 分鐘**

In [ ]:
# Account 1 專用批次校準腳本
CALIB_DIR = "/content/drive/MyDrive/model_quantization/calibration_results"

print("🚀 [1/3] 開始校準 meta-llama/Llama-3.2-1B (~6 分鐘) ...")
!python scripts/calibrate.py --model_id meta-llama/Llama-3.2-1B --output_dir $CALIB_DIR

print("\n🚀 [2/3] 開始校準 google/gemma-2-2b-it (~8 分鐘) ...")
!python scripts/calibrate.py --model_id google/gemma-2-2b-it --output_dir $CALIB_DIR

print("\n🚀 [3/3] 開始校準 meta-llama/Llama-3.2-3B (~17 分鐘) ...")
!python scripts/calibrate.py --model_id meta-llama/Llama-3.2-3B --output_dir $CALIB_DIR

print("\n🎉 Account 1 所有 3 個模型校準全部圓滿完成！")

---
## 🚀 任務排程方案 B：帳號 2 執行 (Account 2)
**目標模型**：`Qwen3.5-2B` ➔ `Qwen3.5-4B`
**總耗時預估**：約 **34 分鐘**

In [ ]:
# Account 2 專用批次校準腳本
CALIB_DIR = "/content/drive/MyDrive/model_quantization/calibration_results"

print("🚀 [1/2] 開始校準 Qwen/Qwen3.5-2B (~11 分鐘) ...")
!python scripts/calibrate.py --model_id Qwen/Qwen3.5-2B --output_dir $CALIB_DIR

print("\n🚀 [2/2] 開始校準 Qwen/Qwen3.5-4B (~23 分鐘) ...")
!python scripts/calibrate.py --model_id Qwen/Qwen3.5-4B --output_dir $CALIB_DIR

print("\n🎉 Account 2 所有 2 個模型校準全部圓滿完成！")

---
## 📊 5. 校準結果檢驗與驗證 (Inspection & Verification)
讀取存放在 Google Drive 中的 `.pt` 快取檔案，列出各層的 Scale 與極端值截斷成效。

In [ ]:
import glob
import os
import torch
import pandas as pd

CALIB_DIR = "/content/drive/MyDrive/model_quantization/calibration_results"
pt_files = sorted(glob.glob(os.path.join(CALIB_DIR, "*.pt")))

print(f"📂 於 Google Drive 找到 {len(pt_files)} 個校準快取檔案：\n")

summary_rows = []
for f in pt_files:
    data = torch.load(f, map_location="cpu")
    m_id = data["model_id"]
    num_layers = data["num_layers"]
    t_sec = data["calibration_time_sec"]
    
    # 抽取第一層與最後一層作為代表
    layer_keys = list(data["layers"].keys())
    first_layer = data["layers"][layer_keys[0]]
    deep_layer = data["layers"][layer_keys[-1]]
    
    summary_rows.append({
        "模型名稱": m_id,
        "受校準層數": num_layers,
        "校準耗時": f"{t_sec:.1f} 秒 ({t_sec/60:.1f} 分)",
        "淺層絕對峰值": f"{first_layer['abs_max']:.2f}",
        "深層絕對峰值": f"{deep_layer['abs_max']:.2f}",
        "深層 99.99% 截斷值": f"{deep_layer['clip_max_9999']:.2f}",
        "深層 INT8 Scale": f"{deep_layer['scales']['int8_sym_scale']:.6f}",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)